# 01 – Cleaning HM Land Registry Price Paid Data (2021)

**Goal:** turn the raw Price Paid file into a clean table of normal residential sales, ready for PostgreSQL, analysis and modelling.

Run the cells from top to bottom. Every cleaning decision is written down with the reason, so you can explain it in an interview.

**Before you start:** put `pp-2021.csv` in `data/raw/`. Do **not** open and save it in Excel.

## Step 1 – Load the file and add the headers

The raw file has **no header row**, so the first sale would be read as column names. We supply the 16 names ourselves (from the gov.uk "Explanations of column headers" page).

We read every column as text (`dtype=str`) first. This stops pandas guessing types, e.g. dropping leading zeros or turning blanks into `NaN` in different ways per column. We convert types deliberately in Step 3.

In [113]:
import pandas as pd
YEAR = 2025
RAW_FILE = f"/Users/yuvarajkotte/PycharmProjects/JupyterProject1/Raw/pp-{YEAR}.csv"
CLEAN_FILE = f"/Users/yuvarajkotte/PycharmProjects/JupyterProject1/Clean/pp-{YEAR}-clean.csv"

COLUMNS = [
    "transaction_id",   # unique ID for each sale
    "price",            # sale price in £
    "date_of_transfer", # date the sale completed
    "postcode",
    "property_type",    # D=Detached, S=Semi-detached, T=Terraced, F=Flat/maisonette, O=Other
    "new_build",        # Y=new build, N=existing property
    "tenure",           # F=Freehold, L=Leasehold
    "paon",             # Primary Addressable Object Name (house number or name)
    "saon",             # Secondary Addressable Object Name (flat number etc.)
    "street",
    "locality",
    "town",
    "district",
    "county",
    "ppd_category",     # A=standard market sale, B=additional (repossessions, buy-to-let, company sales...)
    "record_status",    # A=addition, C=change, D=delete (only matters for monthly update files)
]

df = pd.read_csv(RAW_FILE, header=None, names=COLUMNS, dtype=str, keep_default_na=False)
print(f"Rows loaded: {len(df):,}")
df.head()

Rows loaded: 954,145


,transaction_id,price,date_of_transfer,postcode,property_type,new_build,tenure,paon,saon,street,locality,town,district,county,ppd_category,record_status
0,{42C129E4-C259-60A9-E063-4804A8C0C25D},635000,2025-05-14 00:00,LN11 8GN,D,N,F,2,,KENWICK VIEW,,LOUTH,EAST LINDSEY,LINCOLNSHIRE,A,A
1,{42C129E4-C25B-60A9-E063-4804A8C0C25D},230000,2025-08-29 00:00,PE10 2BL,S,N,F,6,,CONWAY DRIVE,,BOURNE,SOUTH KESTEVEN,LINCOLNSHIRE,A,A
2,{42C129E4-C25E-60A9-E063-4804A8C0C25D},185000,2025-03-27 00:00,LN10 6RJ,S,N,F,6,,KING EDWARD AVENUE,,WOODHALL SPA,EAST LINDSEY,LINCOLNSHIRE,A,A
3,{42C129E4-C260-60A9-E063-4804A8C0C25D},76000,2025-06-06 00:00,NG31 8SP,T,N,L,5,,RAWLE GREEN,,GRANTHAM,SOUTH KESTEVEN,LINCOLNSHIRE,A,A
4,{42C129E4-C261-60A9-E063-4804A8C0C25D},237500,2025-09-18 00:00,PE11 4PP,S,N,F,4,,PLATINUM CLOSE,GOSBERTON,SPALDING,SOUTH HOLLAND,LINCOLNSHIRE,A,A


## Step 2 – Explore data quality before changing anything

Always look first, then decide. We check blanks, the category counts and the price range.

In [114]:
# Blank values per column
blanks = (df == "").sum().to_frame("blank_count")
blanks["blank_pct"] = (blanks["blank_count"] / len(df) * 100).round(1)
blanks

,blank_count,blank_pct
transaction_id,0,0.0
price,0,0.0
date_of_transfer,0,0.0
postcode,2339,0.2
property_type,0,0.0
new_build,0,0.0
tenure,0,0.0
paon,0,0.0
saon,840679,88.1
street,15688,1.6


**What this shows:** `saon` and `locality` are mostly blank. That's normal (most homes aren't flats, and many addresses have no locality), so we keep them and don't treat them as errors. The column that matters is `postcode`: a sale without one can't be matched to a region or location later.

In [115]:
for col in ["property_type", "new_build", "tenure", "ppd_category", "record_status"]:
    print(col, df[col].value_counts().to_dict())

property_type {'T': 264892, 'S': 264425, 'D': 219440, 'F': 157708, 'O': 47680}
new_build {'N': 898686, 'Y': 55459}
tenure {'F': 743720, 'L': 210425}
ppd_category {'A': 794895, 'B': 159250}
record_status {'A': 954145}


In [116]:
# Price summary (converted to numbers just for this check)
pd.to_numeric(df["price"]).describe().apply(lambda x: f"{x:,.0f}")

count        954,145
mean         389,019
std        1,224,986
min                1
25%          186,000
50%          285,000
75%          430,000
max      793,020,000
Name: price, dtype: str

**What this shows:** the minimum price is £1 and the maximum is £900 million. Neither is a normal house sale, so we need to remove extreme values. Also, every row has `record_status = A`, so there are no changes or deletions to apply in a yearly file.

## Step 3 – Fix data types

- `price` → whole number
- `date_of_transfer` → a real date (the time part is always `00:00`, so we drop it)
- blank text → proper missing values (`NA`) so SQL gets `NULL` later

In [117]:
df["price"] = df["price"].astype("int64")
df["date_of_transfer"] = pd.to_datetime(df["date_of_transfer"], format="%Y-%m-%d %H:%M").dt.date
df = df.rename(columns={"date_of_transfer": "sale_date"})

# Turn empty strings into missing values
df = df.replace("", pd.NA)

# Sanity check: every date should be in 2021
print(df["sale_date"].min(), "to", df["sale_date"].max())
df.dtypes

2025-01-01 to 2025-12-31


transaction_id       str
price              int64
sale_date         object
postcode             str
property_type        str
new_build            str
tenure               str
paon                 str
saon                 str
street               str
locality             str
town                 str
district             str
county               str
ppd_category         str
record_status        str
dtype: object

## Step 4 – Apply the cleaning rules

We keep a **cleaning log** so we can report exactly how many rows each rule removed. Put this table in your README.

In [118]:
cleaning_log = [("Raw file", len(df), 0)]

def apply_rule(data, keep_mask, description):
    before = len(data)
    data = data[keep_mask]
    cleaning_log.append((description, len(data), before - len(data)))
    print(f"{description:<55} removed {before - len(data):>8,}   remaining {len(data):>10,}")
    return data

### Rule 1 – Keep only standard market sales (`ppd_category = 'A'`)

Category B includes repossessions, buy-to-let purchases with a mortgage, transfers to companies and other non-standard deals. Their prices don't reflect a normal buyer paying market value, which would confuse the model.

In [119]:
df = apply_rule(df, df["ppd_category"] == "A", "Keep standard market sales (category A)")

Keep standard market sales (category A)                 removed  159,250   remaining    794,895


### Rule 2 – Remove property type 'O' (Other)

"Other" is a mix of things that aren't a normal home (e.g. commercial or mixed-use).

*Interesting finding:* in 2021 **every** 'O' sale is already category B, so Rule 1 removed them all and this rule removes 0 rows. We keep the rule anyway so the code still works on other years. This is a good detail to mention in an interview.

In [120]:
df = apply_rule(df, df["property_type"] != "O", "Remove property type O (Other)")

Remove property type O (Other)                          removed        0   remaining    794,895


### Rule 3 – Remove sales with no postcode

Without a postcode we can't get the region, local authority, coordinates or deprivation score.

In [121]:
df = apply_rule(df, df["postcode"].notna(), "Remove sales with no postcode")

Remove sales with no postcode                           removed      198   remaining    794,697


### Rule 4 – Remove extreme prices (below £10,000 or above £5,000,000)

Prices under £10k are almost never a real open-market home sale (often a share of a property, or a data error). Above £5m is a tiny number of ultra-prime homes that would dominate errors. These cut-offs are a judgement call: say so, and explain why you chose them.

In [122]:
df = apply_rule(df, df["price"].between(10_000, 5_000_000), "Remove prices outside £10k–£5m")

Remove prices outside £10k–£5m                          removed      430   remaining    794,267


### Rule 5 – Remove duplicate sales

`transaction_id` is unique, so there are no exact duplicates. But some sales appear **twice with different IDs**: same address, same date, same price. These are the same sale recorded more than once, so we keep only the first.

In [123]:
print("Duplicate transaction IDs:", df["transaction_id"].duplicated().sum())

dup_key = ["postcode", "paon", "saon", "price", "sale_date"]
print("Same address + date + price:", df.duplicated(subset=dup_key).sum())

# Look at a few examples before deleting
df[df.duplicated(subset=dup_key, keep=False)].sort_values(dup_key).head(6)[
    ["postcode", "paon", "saon", "street", "price", "sale_date", "property_type"]
]

Duplicate transaction IDs: 0
Same address + date + price: 1459


,postcode,paon,saon,street,price,sale_date,property_type
470262,B11 4EY,119,NaN,OAKWOOD ROAD,300000,2025-06-05,D
479130,B11 4EY,119,NaN,OAKWOOD ROAD,300000,2025-06-05,D
508716,B14 5EA,71,NaN,SLADEPOOL FARM ROAD,230000,2025-02-25,S
529525,B14 5EA,71,NaN,SLADEPOOL FARM ROAD,230000,2025-02-25,S
941554,B14 5JS,115,NaN,MAYPOLE LANE,242000,2025-11-25,S
941556,B14 5JS,115,NaN,MAYPOLE LANE,242000,2025-11-25,S


In [124]:
df = apply_rule(df, ~df.duplicated(subset=dup_key, keep="first"), "Remove duplicate sales (same address, date, price)")

Remove duplicate sales (same address, date, price)      removed    1,459   remaining    792,808


## Step 5 – Tidy text and add useful columns

- **postcode_district**: the first half of the postcode (e.g. `E17`). This is what users will type into the app.
- **postcode_area**: the letters only (e.g. `E`, `SW`, `LS`). Useful for bigger groupings.
- **sale_year / sale_month**: for trends and seasonality.
- Readable labels for the one-letter codes, which make charts easier to read.
- Drop `ppd_category` and `record_status`: after cleaning they have only one value, so they carry no information.

In [125]:
df["postcode"] = df["postcode"].str.strip().str.upper()
df["postcode_district"] = df["postcode"].str.split(" ").str[0]
df["postcode_area"] = df["postcode_district"].str.extract(r"^([A-Z]+)", expand=False)

sale_dt = pd.to_datetime(df["sale_date"])
df["sale_year"] = sale_dt.dt.year
df["sale_month"] = sale_dt.dt.month

df["property_type_label"] = df["property_type"].map(
    {"D": "Detached", "S": "Semi-detached", "T": "Terraced", "F": "Flat"})
df["tenure_label"] = df["tenure"].map({"F": "Freehold", "L": "Leasehold"})

df = df.drop(columns=["ppd_category", "record_status"])
df.head()

,transaction_id,price,sale_date,postcode,property_type,new_build,tenure,paon,saon,street,locality,town,district,county,postcode_district,postcode_area,sale_year,sale_month,property_type_label,tenure_label
0,{42C129E4-C259-60A9-E063-4804A8C0C25D},635000,2025-05-14,LN11 8GN,D,N,F,2,NaN,KENWICK VIEW,NaN,LOUTH,EAST LINDSEY,LINCOLNSHIRE,LN11,LN,2025,5,Detached,Freehold
1,{42C129E4-C25B-60A9-E063-4804A8C0C25D},230000,2025-08-29,PE10 2BL,S,N,F,6,NaN,CONWAY DRIVE,NaN,BOURNE,SOUTH KESTEVEN,LINCOLNSHIRE,PE10,PE,2025,8,Semi-detached,Freehold
2,{42C129E4-C25E-60A9-E063-4804A8C0C25D},185000,2025-03-27,LN10 6RJ,S,N,F,6,NaN,KING EDWARD AVENUE,NaN,WOODHALL SPA,EAST LINDSEY,LINCOLNSHIRE,LN10,LN,2025,3,Semi-detached,Freehold
3,{42C129E4-C260-60A9-E063-4804A8C0C25D},76000,2025-06-06,NG31 8SP,T,N,L,5,NaN,RAWLE GREEN,NaN,GRANTHAM,SOUTH KESTEVEN,LINCOLNSHIRE,NG31,NG,2025,6,Terraced,Leasehold
4,{42C129E4-C261-60A9-E063-4804A8C0C25D},237500,2025-09-18,PE11 4PP,S,N,F,4,NaN,PLATINUM CLOSE,GOSBERTON,SPALDING,SOUTH HOLLAND,LINCOLNSHIRE,PE11,PE,2025,9,Semi-detached,Freehold


## Step 6 – Final checks

In [126]:
assert df["transaction_id"].is_unique
assert df["postcode"].notna().all()
assert df["price"].between(10_000, 5_000_000).all()
assert df["property_type_label"].notna().all()
print("All checks passed")

log = pd.DataFrame(cleaning_log, columns=["step", "rows_remaining", "rows_removed"])
log["pct_of_raw_kept"] = (log["rows_remaining"] / log.loc[0, "rows_remaining"] * 100).round(1)
log.to_csv(f"/Users/yuvarajkotte/PycharmProjects/JupyterProject1/Clean/cleaning_log_{YEAR}.csv", index=False)
log

All checks passed


,step,rows_remaining,rows_removed,pct_of_raw_kept
0,Raw file,954145,0,100.0
1,Keep standard market sales (category A),794895,159250,83.3
2,Remove property type O (Other),794895,0,83.3
3,Remove sales with no postcode,794697,198,83.3
4,Remove prices outside £10k–£5m,794267,430,83.2
5,"Remove duplicate sales (same address, date, pr...",792808,1459,83.1


In [127]:
# Median price by property type – a quick check that the numbers look sensible
df.groupby("property_type_label")["price"].agg(["count", "median"]).sort_values("median")

,count,median
property_type_label,,
Flat,128953,240000.0
Terraced,217174,240000.0
Semi-detached,239295,275000.0
Detached,207386,420000.0


## Step 7 – Save the clean file

This file (with a header row) is what you load into PostgreSQL. It lives in `data/clean/`, which `.gitignore` keeps off GitHub.

In [128]:
import os
os.makedirs(os.path.dirname(CLEAN_FILE), exist_ok=True)
df.to_csv(CLEAN_FILE, index=False)
print(f"Saved {len(df):,} rows to {CLEAN_FILE}")

Saved 792,808 rows to /Users/yuvarajkotte/PycharmProjects/JupyterProject1/Clean/pp-2025-clean.csv


In [129]:
   import os
   print(os.path.abspath(CLEAN_FILE))

/Users/yuvarajkotte/PycharmProjects/JupyterProject1/Clean/pp-2025-clean.csv


In [130]:
import os
print(os.path.abspath("data/clean/pp-2021-clean.csv"))
print("File exists:", os.path.exists("data/clean/pp-2021-clean.csv"))

/Users/yuvarajkotte/PycharmProjects/JupyterProject1/uk_house_prices/data/clean/pp-2021-clean.csv
File exists: False


In [131]:
import glob
project = "/Users/yuvarajkotte/PycharmProjects/JupyterProject1"

print("Clean file:", glob.glob(project + "/**/pp-2021-clean.csv", recursive=True))
print("Raw file:  ", glob.glob(project + "/**/pp-2021.csv", recursive=True))

Clean file: ['/Users/yuvarajkotte/PycharmProjects/JupyterProject1/Clean/pp-2021-clean.csv']
Raw file:   ['/Users/yuvarajkotte/PycharmProjects/JupyterProject1/Raw/pp-2021.csv']
